# Basic Self-Attention (PyTorch) - End-to-End Pipeline
**Date**: 2026-05-31  
**Objective**: Implement tokenization, preprocessing, and a basic multi-head self-attention classifier using PyTorch.

## 1) Imports and Setup

In [ ]:
from __future__ import annotations

import math
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}

def load_runtime_env() -> None:
    env_files = [
        Path("configs/runtime.env"),
        Path("configs/runtime.env.example"),
        Path("../configs/runtime.env"),
        Path("../configs/runtime.env.example"),
    ]
    for env_file in env_files:
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            break

def register_local_module_path() -> None:
    candidates = [Path("."), Path("scripts"), Path("self-attention-variants"), Path("self-attention-variants/scripts"), Path("../self-attention-variants"), Path("../self-attention-variants/scripts")]
    for candidate in candidates:
        if (candidate / "shared_text_preprocessing.py").exists():
            resolved = str(candidate.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            return
    raise FileNotFoundError("Could not locate shared_text_preprocessing.py")

load_runtime_env()
register_local_module_path()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))
device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print(f"USE_GPU={int(USE_GPU)} | runtime_device={device}")

from shared_text_preprocessing import PreprocessingConfig, prepare_text_data

## 2) Configuration and Constants

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    num_samples: int = 256
    seq_len: int = 24
    d_model: int = 32
    num_heads: int = 4
    learning_rate: float = 0.05
    epochs: int = 120


CFG = ExperimentConfig()
CFG

## 3) Data Loading

In [ ]:
prep_cfg = PreprocessingConfig(
    num_samples=CFG.num_samples,
    seq_len=CFG.seq_len,
    max_vocab_size=2048,
    train_ratio=0.8,
)
data = prepare_text_data(prep_cfg, seed=SEED)
len(data.texts_train), len(data.texts_test), data.y_train.shape, data.y_test.shape

## 4) EDA

In [ ]:
train_lengths = np.array([len(text.split()) for text in data.texts_train], dtype=np.int64)
class_counts = np.bincount(np.concatenate([data.y_train, data.y_test]), minlength=2)
print("Class counts:", class_counts.tolist())
print("Train token-length mean/std:", float(train_lengths.mean()), float(train_lengths.std()))
print("Sample training sentence:", data.texts_train[0])

## 5) Preprocessing / Feature Engineering

In [ ]:
embedding = torch.nn.Embedding(len(data.vocab), CFG.d_model, device=device)
with torch.no_grad():
    embedding.weight.copy_(torch.randn(len(data.vocab), CFG.d_model, device=device) * 0.2)

train_token_ids_t = torch.tensor(data.train_token_ids, dtype=torch.long, device=device)
test_token_ids_t = torch.tensor(data.test_token_ids, dtype=torch.long, device=device)

x_train = embedding(train_token_ids_t)
x_test = embedding(test_token_ids_t)
print({"vocab_size": len(data.vocab), "x_train_shape": tuple(x_train.shape), "x_test_shape": tuple(x_test.shape)})

## 6) Model Definition

In [ ]:
def init_projection_weights(d_model: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    w_q = torch.randn(d_model, d_model, device=device) * 0.2
    w_k = torch.randn(d_model, d_model, device=device) * 0.2
    w_v = torch.randn(d_model, d_model, device=device) * 0.2
    return w_q, w_k, w_v

def apply_qkv_projection(
    x: torch.Tensor,
    w_q: torch.Tensor,
    w_k: torch.Tensor,
    w_v: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    return x @ w_q, x @ w_k, x @ w_v

def multi_head_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, num_heads: int) -> tuple[torch.Tensor, torch.Tensor]:
    batch, seq_len, d_model = q.shape
    if d_model % num_heads != 0:
        raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
    head_dim = d_model // num_heads
    qh = q.view(batch, seq_len, num_heads, head_dim).permute(0, 2, 1, 3)
    kh = k.view(batch, seq_len, num_heads, head_dim).permute(0, 2, 1, 3)
    vh = v.view(batch, seq_len, num_heads, head_dim).permute(0, 2, 1, 3)
    scores = torch.matmul(qh, kh.transpose(-2, -1)) / math.sqrt(head_dim)
    weights = torch.softmax(scores, dim=-1)
    out = torch.matmul(weights, vh)
    out = out.permute(0, 2, 1, 3).reshape(batch, seq_len, d_model)
    return out, weights

def pooled_features(attn_out: torch.Tensor) -> torch.Tensor:
    return attn_out.mean(dim=1)

## 7) Training

In [ ]:
w_q, w_k, w_v = init_projection_weights(CFG.d_model, device)
q_train, k_train, v_train = apply_qkv_projection(x_train, w_q, w_k, w_v)
q_test, k_test, v_test = apply_qkv_projection(x_test, w_q, w_k, w_v)

attn_train, train_weights = multi_head_attention(q_train, k_train, v_train, CFG.num_heads)
x_feat_train = pooled_features(attn_train).detach()

head = torch.nn.Linear(CFG.d_model, 1, device=device)
optimizer = torch.optim.SGD(head.parameters(), lr=CFG.learning_rate)
y_train_t = torch.tensor(data.y_train, dtype=torch.float32, device=device).view(-1, 1)

for _ in range(CFG.epochs):
    logits = head(x_feat_train)
    loss = F.binary_cross_entropy_with_logits(logits, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training finished.")

## 8) Evaluation and Metrics

In [ ]:
with torch.no_grad():
    attn_test, test_weights = multi_head_attention(q_test, k_test, v_test, CFG.num_heads)
    x_feat_test = pooled_features(attn_test)
    logits_test = head(x_feat_test)
    probs_test = torch.sigmoid(logits_test)
    preds_test = (probs_test >= 0.5).to(torch.int64).view(-1).cpu().numpy()

accuracy = float((preds_test == data.y_test).mean())
metrics = {"accuracy": accuracy}
metrics

## 9) Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Class 0", "Class 1"], class_counts, color=["#4C78A8", "#F58518"])
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("count")

pred_counts = np.bincount(preds_test, minlength=2)
axes[1].bar(["Pred 0", "Pred 1"], pred_counts, color=["#54A24B", "#E45756"])
axes[1].set_title(f"Prediction Distribution | acc={metrics['accuracy']:.3f}")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

## 10) Summary
- Loaded a synthetic raw-text corpus and labels using a shared preprocessing module.
- Converted token IDs to embeddings and trained a PyTorch multi-head self-attention classifier.
- Evaluated test accuracy and visualized label and prediction distributions.